In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM, RDM, CDM
from adapters import Adapter

# Metas

In [4]:
intrinsic_params = ["v", "a", "tau", "s_v", "s_tau", "decay"]

# Priors

In [5]:
priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

# Context Manager

In [6]:
context_manager = ContextManager()

# Model Family

In [7]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=priors,
    intrinsic_params=intrinsic_params,
)

In [8]:
samples = model_family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500
)

In [9]:
samples["param_masks"].shape

(3, 180)

In [10]:
samples["param_matrices"].shape

(3, 180)

In [11]:
samples["regressor_masks"].shape

(3, 30)

In [12]:
samples

{'model_names': ['DDM', 'DDM', 'DDM'],
 'design_configs': [{'u_0': ['a', 'decay'],
   'u_1': ['a', 'tau', 's_v'],
   'u_2': ['tau', 's_v', 'decay'],
   'u_3': ['v', 'a', 'tau'],
   'u_4': ['v', 'a', 's_v'],
   'u_5': ['a', 'decay']},
  {'u_0': ['a', 's_v', 'decay'],
   'u_1': ['v'],
   'u_2': ['v', 's_v', 'decay'],
   'u_3': ['a'],
   'u_4': ['v', 'a', 'decay'],
   'u_5': ['v', 'a'],
   'u_6': ['v', 'a', 'decay'],
   'u_7': ['v', 's_v'],
   'u_8': ['v', 'tau', 's_v', 'decay']},
  {'u_0': ['v', 'decay'],
   'u_1': ['a', 'tau', 's_v'],
   'u_2': ['v', 'a'],
   'u_3': ['a', 's_v', 'decay'],
   'u_4': ['v'],
   'u_5': ['a', 'decay'],
   'u_6': ['v', 'a', 'decay'],
   'u_7': ['tau', 's_v'],
   'u_8': ['v', 'tau'],
   'u_9': ['a', 'tau']}],
 'design_matrices': array([[[1.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [1.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.    

# Adapter

In [13]:
adapter = Adapter()

In [14]:
design_matrices = adapter.convert_dtype(samples["design_matrices"], dtype=np.float32)
param_masks = adapter.convert_dtype(samples["param_masks"], dtype=np.float32)
rts = adapter.convert_dtype(samples["sim_data"]["rts"], dtype=np.float32)
choices = adapter.convert_dtype(samples["sim_data"]["choices"], dtype=np.float32)

In [15]:
batch_size, num_obs, num_cols = design_matrices.shape
print(batch_size, num_obs, num_cols)

3 341 30


In [16]:
y_rts_col = adapter.atleast_2d(rts, orientation="col")                 # (N, 1)
y_ch_col  = adapter.atleast_2d(rts,  orientation="col")

In [17]:
sim_data = adapter.concatenate([y_rts_col, y_ch_col], axis=1, dtype=np.float32, pad=False)

# RDM

In [18]:
rdm_priors = {
    "v":      {"intercept": lambda: np.random.gamma(3.0, 0.8),
               "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":      {"intercept": lambda: np.random.gamma(10.0, 0.3),
               "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":    {"intercept": lambda: np.random.gamma(3.0, 0.2),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
    "decay":  {"intercept": lambda: np.random.gamma(1.0, 0.4),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
}

In [19]:
family = NestedModelFamily(
    name="RDM",
    model=RDM(),
    context_manager=context_manager,
    prior_fun=rdm_priors,
    intrinsic_params=intrinsic_params,
)

In [20]:
num_alternatives = np.random.randint(2, 4, size=1)  # number of alternatives

context = {
    "correct_idx": lambda n: np.random.randint(0, num_alternatives, size=n),
    "num_alternatives": num_alternatives,
}

In [22]:
samples = family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau"},
        fixed_intrinsics={"decay"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    context=context,
)

In [23]:
samples

{'model_names': ['RDM', 'RDM', 'RDM'],
 'design_configs': [{'u_0': ['v', 'tau'],
   'u_1': ['a'],
   'u_2': ['a'],
   'u_3': ['a', 'tau']},
  {'u_0': ['v', 'tau'], 'u_1': ['v', 'a', 'tau'], 'u_2': ['a']},
  {'u_0': ['v'],
   'u_1': ['a', 'tau'],
   'u_2': [],
   'u_3': ['v', 'a'],
   'u_4': ['tau'],
   'u_5': [],
   'u_6': ['v', 'a'],
   'u_7': ['tau']}],
 'design_matrices': array([[[0.22698515, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.77845248, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.915932  , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.8703955 , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.31923023, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.82660039, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]],
 
        [[0.      